# 🪙 Smart Donation Box: YOLOv8 Currency Detection & Training Pipeline
An end-to-end notebook for **Exploratory Data Analysis (EDA)**, **YOLOv8-nano Training**, **Model Validation**, and **Deployment Synchronization** on the Indian Currency Dataset.


## 1. Environment Setup & Library Imports


In [ ]:
import os
import glob
import json
import shutil
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
from ultralytics import YOLO

# Set style
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.sans-serif"] = "Arial"

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Dataset Definition & Structure Inspection
The Indian Currency dataset consists of 6 primary denominations: **₹10, ₹20, ₹50, ₹100, ₹200, ₹500** distributed across `train`, `valid`, and `test` splits.


In [ ]:
# Define Dataset Paths & Class Mappings
WORKSPACE_DIR = os.getcwd()
DATASET_DIR = os.path.join(WORKSPACE_DIR, "data", "Indian currency")

CLASS_NAMES = {
    0: "10",
    1: "100",
    2: "20",
    3: "200",
    4: "50",
    5: "500"
}

CLASS_COLORS = {
    "10": "#2d52a0",
    "20": "#32cd32",
    "50": "#ffbf00",
    "100": "#d355ba",
    "200": "#008cff",
    "500": "#808080"
}

print(f"Dataset root: {DATASET_DIR}")
print("Classes mapping:", CLASS_NAMES)


## 3. Exploratory Data Analysis (EDA)


### 3.1. Parsing Annotations & Image Dimensions into a DataFrame


In [ ]:
def parse_dataset(dataset_dir, class_names):
    records = []
    splits = ["train", "valid", "test"]
    
    for split in splits:
        img_dir = os.path.join(dataset_dir, split, "images")
        lbl_dir = os.path.join(dataset_dir, split, "labels")
        
        img_paths = glob.glob(os.path.join(img_dir, "*.*"))
        for img_p in img_paths:
            stem = os.path.splitext(os.path.basename(img_p))[0]
            lbl_p = os.path.join(lbl_dir, stem + ".txt")
            
            try:
                with Image.open(img_p) as im:
                    w, h = im.size
            except Exception:
                w, h = 640, 640
                
            if os.path.exists(lbl_p):
                with open(lbl_p, "r") as f:
                    lines = [line.strip() for line in f if line.strip()]
                    if not lines:
                        records.append({
                            "split": split, "img_name": os.path.basename(img_p),
                            "img_path": img_p, "width": w, "height": h,
                            "class_id": -1, "class_name": "Background",
                            "cx": 0, "cy": 0, "bw": 0, "bh": 0, "box_area": 0, "aspect_ratio": 0
                        })
                    for line in lines:
                        parts = line.split()
                        if len(parts) >= 5:
                            cid = int(parts[0])
                            cx, cy, bw, bh = map(float, parts[1:5])
                            records.append({
                                "split": split,
                                "img_name": os.path.basename(img_p),
                                "img_path": img_p,
                                "width": w,
                                "height": h,
                                "class_id": cid,
                                "class_name": class_names.get(cid, str(cid)),
                                "cx": cx, "cy": cy, "bw": bw, "bh": bh,
                                "box_area": bw * bh,
                                "aspect_ratio": (bw * w) / (bh * h) if (bh * h) > 0 else 0
                            })
    return pd.DataFrame(records)

df = parse_dataset(DATASET_DIR, CLASS_NAMES)
print(f"Total parsed bounding box annotations: {len(df)}")
df.head()


### 3.2. Class Distribution Across Splits


In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df[df['class_name'] != 'Background'], x='class_name', hue='split', palette='Blues_d', order=['10', '20', '50', '100', '200', '500'])
plt.title("Denomination Distribution across Train, Valid, and Test Splits", fontsize=14, fontweight='bold')
plt.xlabel("Currency Denomination (₹)", fontsize=12)
plt.ylabel("Number of Annotations", fontsize=12)
plt.legend(title="Split")
plt.tight_layout()
plt.show()

# Table representation
split_class_pivot = pd.crosstab(df['class_name'], df['split'])
print("--- Class Frequencies ---")
print(split_class_pivot)


### 3.3. Bounding Box Geometry Analysis (Area & Aspect Ratio)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box Area Distribution
sns.histplot(data=df[df['class_name'] != 'Background'], x='box_area', hue='class_name', multiple='stack', ax=axes[0], palette='turbo')
axes[0].set_title("Normalized Bounding Box Area Distribution", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Relative Area (Normalized: w * h)")

# Aspect Ratio Distribution
sns.boxplot(data=df[df['class_name'] != 'Background'], x='class_name', y='aspect_ratio', ax=axes[1], palette='Set2', order=['10', '20', '50', '100', '200', '500'])
axes[1].set_title("Aspect Ratio (Width / Height) by Denomination", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Denomination (₹)")
axes[1].set_ylabel("Aspect Ratio")

plt.tight_layout()
plt.show()


### 3.4. Sample Ground Truth Visualizer


In [ ]:
def visualize_sample_annotations(df, num_samples=6):
    samples = df[df['class_name'] != 'Background'].drop_duplicates(subset=['img_path']).sample(num_samples, random_state=42)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, (_, row) in enumerate(samples.iterrows()):
        img_p = row['img_path']
        img = cv2.imread(img_p)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        # Draw all boxes for this image
        img_boxes = df[df['img_path'] == img_p]
        for _, box in img_boxes.iterrows():
            if box['class_name'] == 'Background':
                continue
            xmin = int((box['cx'] - box['bw'] / 2) * w)
            ymin = int((box['cy'] - box['bh'] / 2) * h)
            xmax = int((box['cx'] + box['bw'] / 2) * w)
            ymax = int((box['cy'] + box['bh'] / 2) * h)
            
            label = f"₹{box['class_name']}"
            cv2.rectangle(img, (xmin, ymin), (xmax, ymax), (0, 255, 0), 3)
            cv2.putText(img, label, (xmin + 5, ymin + 25), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
            
        axes[idx].imshow(img)
        axes[idx].set_title(f"Image: {row['img_name']} ({row['split']})", fontsize=11)
        axes[idx].axis("off")
        
    plt.tight_layout()
    plt.show()

visualize_sample_annotations(df, num_samples=6)


## 4. Dataset Configuration (data.yaml Generation)
We dynamically generate the `data.yaml` configuration ensuring proper POSIX absolute paths.


In [ ]:
yaml_path = os.path.join(DATASET_DIR, "data.yaml")
posix_dataset_dir = DATASET_DIR.replace(os.sep, "/")

yaml_content = f"""path: "{posix_dataset_dir}"
train: train/images
val: valid/images
test: test/images

names:
  0: "10"
  1: "100"
  2: "20"
  3: "200"
  4: "50"
  5: "500"
"""

with open(yaml_path, "w", encoding="utf-8") as f:
    f.write(yaml_content)

print(f"Generated data.yaml at: {yaml_path}")


## 5. Model Architecture & Hyperparameter Rules
### Banknote Training Strategy:
1. **Model Backbone**: `yolov8n.pt` (Nano) provides optimal speed-accuracy tradeoff for embedded kiosks (~3.2M parameters).
2. **Augmentations**:
   - `degrees=10.0`: Handles slight banknote rotations when inserted into the donation slot.
   - `hsv_s=0.7`, `hsv_v=0.4`: Simulates lighting fluctuations and faded/soiled currency notes.
   - `mosaic=0.7`, `mixup=0.1`: Boosts contextual multi-scale detection.
   - `close_mosaic=10`: Stops mosaic during the final 10 epochs for crisp bounding box convergence.
3. **Optimizer**: `AdamW` with `lr0=0.001` and Cosine Learning Rate schedule.


In [ ]:
# Initialize YOLOv8 Model
base_model = YOLO("yolov8n.pt")
device = 0 if torch.cuda.is_available() else "cpu"
print(f"Training on device: {device}")


In [ ]:
# Run YOLOv8 Training
EPOCHS = 100
BATCH_SIZE = 16
IMG_SIZE = 640

training_results = base_model.train(
    data=yaml_path,
    epochs=EPOCHS,
    patience=25,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=device,
    workers=0,                 # Windows compatibility
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    cos_lr=True,
    warmup_epochs=3.0,
    # Augmentation rules
    augment=True,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.4,
    shear=2.0,
    perspective=0.0005,
    fliplr=0.5,
    mosaic=0.7,
    mixup=0.1,
    close_mosaic=10,
    # Outputs
    project=os.path.join(WORKSPACE_DIR, "runs", "train"),
    name="indian_currency_yolov8",
    exist_ok=True,
    save=True,
    verbose=True
)


## 6. Performance Evaluation & Validation


In [ ]:
# Evaluate Model on Test/Valid set
best_weights = os.path.join(WORKSPACE_DIR, "runs", "train", "indian_currency_yolov8", "weights", "best.pt")
trained_model = YOLO(best_weights)

val_metrics = trained_model.val(data=yaml_path, imgsz=IMG_SIZE, device=device)

map50 = getattr(val_metrics.box, 'map50', 0.0)
map50_95 = getattr(val_metrics.box, 'map', 0.0)

print(f"📈 Final Validation mAP@0.50:      {map50:.4f} ({map50*100:.2f}%)")
print(f"📈 Final Validation mAP@0.50:0.95: {map50_95:.4f} ({map50_95*100:.2f}%)")


### 6.1. Visualizing Training Results & Metric Curves


In [ ]:
results_img_path = os.path.join(WORKSPACE_DIR, "runs", "train", "indian_currency_yolov8", "results.png")
if os.path.exists(results_img_path):
    results_img = Image.open(results_img_path)
    plt.figure(figsize=(16, 10))
    plt.imshow(results_img)
    plt.axis("off")
    plt.title("YOLOv8 Training Curves & Loss Progression", fontsize=14, fontweight='bold')
    plt.show()
else:
    print("Training summary image not found yet.")


### 6.2. Test Inference & Visual Verification


In [ ]:
# Run inference on Test images
test_img_dir = os.path.join(DATASET_DIR, "test", "images")
test_samples = glob.glob(os.path.join(test_img_dir, "*.*"))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, img_p in enumerate(test_samples):
    res = trained_model.predict(img_p, imgsz=IMG_SIZE, conf=0.40, verbose=False)[0]
    res_bgr = res.plot()
    res_rgb = cv2.cvtColor(res_bgr, cv2.COLOR_BGR2RGB)
    
    axes[idx].imshow(res_rgb)
    axes[idx].set_title(f"Test: {os.path.basename(img_p)}", fontsize=11)
    axes[idx].axis("off")

plt.tight_layout()
plt.show()


## 7. Model Synchronization with detector.py & app.py
Synchronizing the best model weights to `models/yolov8n_currency_best.pt` and exporting to ONNX format.


In [ ]:
# 1. Copy weights to models/ folder
models_dir = os.path.join(WORKSPACE_DIR, "models")
os.makedirs(models_dir, exist_ok=True)
dest_pt_path = os.path.join(models_dir, "yolov8n_currency_best.pt")

shutil.copy(best_weights, dest_pt_path)
print(f"✅ Aligned PyTorch weights saved to: {dest_pt_path}")

# 2. Export ONNX model
try:
    onnx_file = trained_model.export(format="onnx", imgsz=IMG_SIZE, dynamic=False)
    dest_onnx_path = os.path.join(models_dir, "yolov8n_currency_best.onnx")
    if os.path.exists(onnx_file):
        shutil.copy(onnx_file, dest_onnx_path)
        print(f"✅ Exported ONNX model saved to: {dest_onnx_path}")
except Exception as e:
    print(f"⚠️ ONNX export note: {e}")

# 3. Update models/config.json
config_file = os.path.join(models_dir, "config.json")
config_data = {}
if os.path.exists(config_file):
    try:
        with open(config_file, "r") as f:
            config_data = json.load(f)
    except Exception:
        pass

config_data.update({
    "architecture": "YOLOv8-nano + MobileNetV3-Small",
    "num_classes": len(CLASS_NAMES),
    "class_names": list(CLASS_NAMES.values()),
    "yolo_img_size": IMG_SIZE,
    "yolo_map50": round(float(map50), 4),
    "yolo_map50_95": round(float(map50_95), 4),
    "trained_model_file": "yolov8n_currency_best.pt"
})

with open(config_file, "w") as f:
    json.dump(config_data, f, indent=2)

print(f"✅ Updated metadata in: {config_file}")
print("\n🎉 Workflow complete! Launch app with: streamlit run app.py")
